# 00 · Colab setup
Rebuilds a working SecureMCP session on a free Colab T4. Use **Runtime → Run all**.
Steps: GPU check → mount Drive → clone repo → install deps → cache embedding model → smoke test → (optional) real-model Phase 1 check.

**Private repo?** Add a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar,
a fine-grained token with read access to the repo) and enable *Notebook access*.

In [ ]:
# 1. GPU check
import subprocess, shutil
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
else:
    print("No GPU. Runtime → Change runtime type → T4 GPU, then Run all again.")

In [ ]:
# 2. Mount Google Drive and create the persistent folders
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/securemcp"
for sub in ["data", "models", "llm_cache", "results", "hf_cache"]:
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

# Keep downloaded Hugging Face models on Drive so they survive disconnects
os.environ["HF_HOME"] = f"{DRIVE_ROOT}/hf_cache"
print("Drive ready:", DRIVE_ROOT)

In [ ]:
# 3. Clone (or update) the repo
import os, subprocess
REPO_OWNER = "RajyalakshmiKillamsetty2023"
REPO_NAME = "securemcp"
REPO_DIR = f"/content/{REPO_NAME}"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

url = (f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git" if token
       else f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git")

if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

# Do not leave the token in the saved remote URL
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin",
                f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-3"],
                     capture_output=True, text=True).stdout)

In [ ]:
# 4. Install dependencies
# Colab already ships torch with CUDA, so we install only what the project needs.
# (No editable install: Colab's Python version may be newer than the laptop's 3.11 pin.)
%pip install -q sentence-transformers scikit-learn pandas datasets \
    pydantic pydantic-settings pyyaml httpx

import sys
sys.path.insert(0, f"{REPO_DIR}/backend")

In [ ]:
# 5. Download and cache the embedding model (~90 MB, saved to Drive)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Loaded on:", model.device)

In [ ]:
# 6. Smoke test
from sentence_transformers import util
from app.config import settings

q = "What is the weather in Hyderabad tomorrow?"
good = "Returns the weather forecast for a city."
bad = "Converts a PDF file to Word format."
emb = model.encode([q, good, bad], convert_to_tensor=True)
s_good = float(util.cos_sim(emb[0], emb[1]))
s_bad = float(util.cos_sim(emb[0], emb[2]))
print(f"relevant={s_good:.3f}  irrelevant={s_bad:.3f}")
assert s_good > s_bad, "Embedding sanity check failed"
print("Config loaded, mongo_db =", settings.mongo_db)
print("\nSetup OK. Drive root:", DRIVE_ROOT)

## Phase 1 check: real model through Ollama
Installs Ollama, pulls the model, runs four queries through the agent twice, and confirms the second
pass is served from the cache on Drive. The first run takes about 5 to 8 minutes (model download is
about 4.7 GB). Set `RUN_OLLAMA_CHECK = False` to skip it when you only need the environment.

The model itself is not stored on Drive: re-pulling is quick on Colab and large files on Drive are slow.

In [ ]:
RUN_OLLAMA_CHECK = True
OLLAMA_MODEL = "qwen2.5:7b"   # tag from the Qwen docs; untagged Qwen2.5 variants are already instruct models

if RUN_OLLAMA_CHECK:
    import shutil, subprocess, time, urllib.request

    if not shutil.which("ollama"):
        subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null", shell=True, check=True)
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

    def ollama_up():
        try:
            urllib.request.urlopen("http://127.0.0.1:11434", timeout=2)
            return True
        except Exception:
            return False

    if not ollama_up():
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(30):
            if ollama_up():
                break
            time.sleep(1)
    assert ollama_up(), "Ollama server did not start"

    subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)
    print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

In [ ]:
if RUN_OLLAMA_CHECK:
    from app.agent.agent import Agent
    from app.config import Settings
    from app.llm.factory import build_llm
    from app.schemas import AgentRequest

    s = Settings(llm_mode="ollama", llm_model=OLLAMA_MODEL,
                 llm_cache_path=f"{DRIVE_ROOT}/llm_cache/phase1_check.jsonl")
    llm = build_llm(s)
    agent = Agent(llm)

    CASES = [
        ("What is the weather in Hyderabad today?", "get_weather"),
        ("What is 15 * 4?", "calculator"),
        ("Convert 100 USD to INR", "convert_currency"),
        ("Find flights from Delhi to Mumbai on 2026-12-01", "search_flights"),
    ]

    def run_pass(label):
        entries_before, t0, rows = len(llm.cache), time.perf_counter(), []
        for query, expected in CASES:
            r = agent.run(AgentRequest(query=query))
            ran_ok = bool(r.executed and r.result and r.result.ok)
            rows.append((query, expected, r.call.tool, ran_ok, r.call.parse_error))
        secs = time.perf_counter() - t0
        added = len(llm.cache) - entries_before
        print(f"\n{label}: {secs:.1f}s, {added} new cache entries")
        for q, exp, got, ran_ok, err in rows:
            flag = "OK " if (got == exp and ran_ok) else "BAD"
            print(f"  [{flag}] {q}\n        expected={exp} got={got} ran={ran_ok} parse_error={err}")
        return rows, added

    rows1, added1 = run_pass("Pass 1 (model)")
    rows2, added2 = run_pass("Pass 2 (should come from cache)")

    correct = all(got == exp and ran_ok for _, exp, got, ran_ok, _ in rows1)
    from_cache = added2 == 0
    print("\nRight tool chosen and run for every query:", correct)
    print("Second pass served entirely from cache:  ", from_cache)
    print("Phase 1 exit check:", "PASS" if (correct and from_cache) else "NOT YET (see rows above)")